# CareerPilot AI — 2. Job Fetching

Pulls live postings from the Adzuna API and saves them to `job_dataset.json`.

**Fixed in this version:** search terms used to be a hardcoded list, so every candidate
got the exact same 34 generic jobs regardless of their resume. This notebook now reads
the most recently parsed profile from `careerpilot.db` and builds the Adzuna search
terms from that candidate's own skills / career goal. If no profile is found, it falls
back to the old generic terms so the notebook still runs standalone.

**It still doesn't require `1_resume_parsing.ipynb` to have run first** — it *tries* to
read `careerpilot.db` for personalization, but works without it too.

**Run this once ahead of your demo, not live during it.** Re-run it whenever a new resume
is parsed, so the job pool actually reflects the candidate — but you're on Trial Access,
so don't hammer the API while testing.


## Step 1 — Setup

In [6]:
!pip install requests --quiet
import json
import os
import sqlite3
from datetime import datetime, timezone

import requests

DB_PATH = "careerpilot.db"
JOB_DATASET_PATH = "job_dataset.json"


## Step 2 — Skill tagging

A plain keyword list — the **same list** used in `1_resume_parsing.ipynb`'s Step 4 and
in `5_skill_gap_analysis.ipynb`, kept in sync by hand so job skills, profile skills, and
gap analysis all use identical vocabulary.

**Fixed in this version:** this list had drifted out of sync with what
`1_resume_parsing.ipynb` actually extracts — real parsed profiles in `careerpilot.db`
contain skills like `MySQL`, `REST API`, and `Spring Boot` that weren't in this list, so
they could never be matched against job postings. Added the missing common ones below.
If you add a new skill anywhere, add it to **all three** notebooks.


In [7]:
SKILLS = [
    "Python", "Java", "JavaScript", "TypeScript", "SQL", "MySQL", "PostgreSQL", "MongoDB",
    "React", "Angular", "Vue", "Node.js", "FastAPI", "Flask", "Django", "Spring Boot",
    "REST API", "GraphQL",
    "Machine Learning", "Deep Learning", "NLP", "Data Analysis",
    "AWS", "Azure", "GCP", "Docker", "Kubernetes", "CI/CD", "Linux",
    "Git", "spaCy", "TensorFlow", "PyTorch",
    "HTML", "CSS", "C++", "C#", ".NET", "R",
    "Excel", "Tableau", "Power BI",
]

def get_skills(text: str) -> list:
    # simple case-insensitive substring match — no spaCy dependency in this notebook
    text_lower = text.lower()
    return sorted({skill for skill in SKILLS if skill.lower() in text_lower})

print(f"Skill tagger ready ({len(SKILLS)} tracked skills).")


Skill tagger ready (42 tracked skills).


## Step 3 — Build search terms from the candidate's profile

Reads the most recently saved profile from `careerpilot.db` and turns their skills and
career goal into Adzuna search terms, instead of always searching the same four generic
roles. Falls back to generic terms if there's no profile yet (e.g. first run, or
`1_resume_parsing.ipynb` hasn't been run).


In [8]:
FALLBACK_SEARCH_TERMS = ["python developer", "data analyst", "software engineer", "machine learning"]
MAX_SEARCH_TERMS = 5

def load_latest_profile(db_path: str) -> dict | None:
    if not os.path.exists(db_path):
        return None
    conn = sqlite3.connect(db_path)
    try:
        cur = conn.cursor()
        cur.execute("SELECT * FROM career_profiles ORDER BY id DESC LIMIT 1")
        row = cur.fetchone()
        if row is None:
            return None
        cols = [d[0] for d in cur.description]
    finally:
        conn.close()
    profile = dict(zip(cols, row))
    profile["skills"] = json.loads(profile["skills"]) if profile["skills"] else []
    return profile

def extract_career_goal_text(raw_goal) -> str:
    """career_goal is stored as a JSON string (often just "{}" if the parser found
    nothing usable) — pull out a readable target-role phrase if one exists."""
    if not raw_goal:
        return ""
    try:
        parsed = json.loads(raw_goal)
    except (json.JSONDecodeError, TypeError):
        return str(raw_goal).strip()
    if isinstance(parsed, dict):
        for key in ("target_role", "desired_role", "role", "goal", "title"):
            if parsed.get(key):
                return str(parsed[key]).strip()
        return ""
    return str(parsed).strip()

def build_search_terms(profile: dict | None) -> tuple[list, int | None]:
    if profile is None:
        return FALLBACK_SEARCH_TERMS, None

    terms = []
    goal_text = extract_career_goal_text(profile.get("career_goal"))
    if goal_text:
        terms.append(goal_text)
    terms.extend(profile.get("skills", []))

    # de-dupe, preserve order
    seen = set()
    deduped = []
    for t in terms:
        key = t.lower().strip()
        if key and key not in seen:
            seen.add(key)
            deduped.append(t)

    if not deduped:
        # Parsed profile exists but has no usable skills/goal (e.g. skills=[] because
        # the resume's vocabulary fell outside the tracked SKILLS list) — fall back
        # rather than searching Adzuna with nothing.
        return FALLBACK_SEARCH_TERMS, profile["id"]

    return deduped[:MAX_SEARCH_TERMS], profile["id"]

profile = load_latest_profile(DB_PATH)
ADZUNA_SEARCH_TERMS, profile_id = build_search_terms(profile)

if profile is None:
    print("No profile found in careerpilot.db — using generic fallback search terms.")
    print(f"  {ADZUNA_SEARCH_TERMS}")
    print("Run 1_resume_parsing.ipynb first to fetch jobs tailored to a specific candidate.")
else:
    print(f"Candidate: profile #{profile_id} — {profile.get('name') or profile.get('filename')}")
    print(f"Search terms built from this profile: {ADZUNA_SEARCH_TERMS}")


Candidate: profile #5 — linkedin.com/in/sivateja-somisetty •
Search terms built from this profile: ['CSS', 'FastAPI', 'Git', 'Java', 'JavaScript']


## Step 4 — Fetch from Adzuna

Searches the terms built above, de-duplicates overlapping results, tags skills, and
keeps the direct application link.


In [9]:
# --- Adzuna credentials ---
# Keep these out of anything you commit publicly (e.g. a public GitHub repo).
ADZUNA_APP_ID = "6abd39fa"
ADZUNA_APP_KEY = "1a615aa69c7325a614caa450b165077f"
ADZUNA_COUNTRY = "in"   # Adzuna also supports gb, us, au, ca, de, fr, and others
RESULTS_PER_TERM = 10

def fetch_adzuna_jobs(search_term: str, country: str = ADZUNA_COUNTRY, results: int = RESULTS_PER_TERM) -> list:
    url = f"https://api.adzuna.com/v1/api/jobs/{country}/search/1"
    params = {
        "app_id": ADZUNA_APP_ID,
        "app_key": ADZUNA_APP_KEY,
        "results_per_page": results,
        "what": search_term,
        "content-type": "application/json",
    }
    response = requests.get(url, params=params, timeout=15)
    response.raise_for_status()   # fail loudly on a bad key / bad request
    return response.json().get("results", [])

def normalize_adzuna_job(raw: dict) -> dict:
    description = raw.get("description", "")
    return {
        "title": raw.get("title", "Untitled"),
        "company": raw.get("company", {}).get("display_name", "Unknown"),
        "location": raw.get("location", {}).get("display_name", ""),
        "skills": get_skills(description),
        "description": description,
        "apply_link": raw.get("redirect_url", ""),
    }

all_jobs = []
seen = set()

for term in ADZUNA_SEARCH_TERMS:
    raw_results = fetch_adzuna_jobs(term)
    for raw in raw_results:
        job = normalize_adzuna_job(raw)
        key = (job["title"], job["company"])
        if key not in seen:
            seen.add(key)
            all_jobs.append(job)

print(f"Fetched {len(all_jobs)} unique job postings using terms: {ADZUNA_SEARCH_TERMS}")


Fetched 44 unique job postings using terms: ['CSS', 'FastAPI', 'Git', 'Java', 'JavaScript']


## Step 5 — Save to job_dataset.json

**Fixed in this version:** `job_dataset.json` used to just be a bare list of jobs with
no link back to whichever profile it was fetched for, so there was no way to tell it
had gone stale after a new resume was parsed. It now stores the `profile_id` and a
timestamp alongside the jobs, mirroring the same freshness check
`5_skill_gap_analysis.ipynb` already does for `top_matches.json`.
`3_job_matching.ipynb` reads this next and will warn if it's stale for the current
profile.


In [10]:
output = {
    "profile_id": profile_id,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "search_terms": ADZUNA_SEARCH_TERMS,
    "jobs": all_jobs,
}

with open(JOB_DATASET_PATH, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved {JOB_DATASET_PATH} ({len(all_jobs)} jobs, profile #{profile_id}) — ready for 3_job_matching.ipynb")


Saved job_dataset.json (44 jobs, profile #5) — ready for 3_job_matching.ipynb
